# 探索性概览

基于主面板快速浏览全球健康结果、资源和疾病结构。


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if ROOT.name == "notebooks" and (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError("无法定位仓库根目录，请从项目根目录或 notebooks/ 目录启动 Notebook。")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)

def read_api(relpath: str):
    payload = read_json(ROOT / relpath)
    return payload.get("data", payload)

def show_image(relpath: str, width: int = 900):
    display(Image(filename=str(ROOT / relpath), width=width))

def require(*relpaths: str):
    missing = [path for path in relpaths if not (ROOT / path).exists()]
    if missing:
        raise FileNotFoundError(
            "缺少以下产物，请先运行 `PYTHONPATH=src python -m hdi.data.integrator` 和 "
            "`PYTHONPATH=src python -m hdi.analysis.competition`:\n- " + "\n- ".join(missing)
        )


In [ ]:
require("data/processed/master_panel.parquet", "api_output/metadata/summary.json")
master = pd.read_parquet(ROOT / "data/processed/master_panel.parquet")
summary = read_api("api_output/metadata/summary.json")

display(pd.Series(summary, name="summary").to_frame())
display(master[["life_expectancy", "communicable_share", "ncd_share", "gdp_per_capita"]].describe().T)


In [ ]:
latest = master[master["year"] == master["year"].max()].copy()
latest = latest.sort_values("life_expectancy", ascending=False)
display(latest[["country_name", "who_region", "life_expectancy", "communicable_share", "ncd_share"]].head(15))
